# UNITER / VILLA-ITM v2 — Model Evaluation on New Test Set

Loads the saved `best_model.pt` checkpoint, rebuilds the exact model and visual encoder used during training, runs inference on a new test CSV, and prints a full suite of classification metrics.

**Stream recap**
- Stream A (ITM core): `desc` + image region tokens → `itm_proj` → 256-dim
- Stream B (text-only): `caption` + YOLO tokens → `caption_proj` → 128-dim
- FairFace metadata → 16-dim
- Classifier head: concat(256, 128, 16) = 400-dim → logit

## Cell 1 — Imports

In [1]:
import os
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import AutoTokenizer, BertModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt
import seaborn as sns

# ── Make the training script importable ──────────────────────────────────────
# Point this at the directory containing UNITER_VILLA_ITM_v2.py
SCRIPT_DIR = os.path.expanduser("~")   # adjust if the script lives elsewhere
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

from Uniter_desc_Villa import (
    UNITERITMClassifier,
    ResNetRegionExtractor,
    UNITERBinaryDataset,
    build_object_token_string,
    build_fairface_vector,
)

print("Imports OK")

Imports OK


## Cell 2 — Configuration

Edit the paths and hyperparameters below to match your training run.
The hyperparameters must be identical to the values used during training —
they define the model architecture that the saved weights expect.

In [2]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
CHECKPOINT_PATH  = os.path.expanduser("New folder/uniter_desc_checkpoints/best_model.pt")

# ── New test data ─────────────────────────────────────────────────────────────
TEST_CAPTIONS_CSV  = os.path.expanduser("./merged_dataset.csv")   # filename, text, label
TEST_OBJECTS_CSV   = os.path.expanduser("./object_detections.csv")
TEST_FACES_CSV     = os.path.expanduser("./human_results.csv")
TEST_DESC_CSV      = os.path.expanduser("./desc.csv")                # filename, desc
IMAGE_DIR          = os.path.expanduser("./clean3")

# ── Model hyperparameters (must match training) ───────────────────────────────
PRETRAINED_NAME  = "bert-base-uncased"   # or local UNITER/VILLA path
RESNET_WEIGHTS   = os.path.expanduser("~/resnet50_imagenet.pth")
IMG_SIZE         = 224
MAX_TEXT_LEN     = 128    # caption + YOLO tokens
MAX_DESC_LEN     = 128    # desc tokens
MAX_VISUAL_LEN   = 36
HIDDEN_DIM       = 256    # Stream A itm_proj output
CAPTION_DIM      = 128    # Stream B caption_proj output
DROPOUT          = 0.3
FREEZE_LAYERS    = 6

# ── Inference ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 16
NUM_WORKERS  = 4
THRESHOLD    = 0.5        # sigmoid threshold for hard predictions

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## Cell 3 — Rebuild Models and Load Checkpoint

In [3]:
# ── Load checkpoint ───────────────────────────────────────────────────────────
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
print(f"Checkpoint from epoch {ckpt['epoch']} | val_f1={ckpt['val_f1']:.4f}")

sd = ckpt["model_state"]

# ── Infer exact architecture dims from the saved weights ─────────────────────
# This makes loading robust to version differences — the weights define the
# true shape; we never have to guess or match a config manually.
#
# classifier.0.weight : (128, fused_dim)  where fused_dim = hidden_dim + caption_dim + 16
# itm_proj.0.weight   : (hidden_dim, bert_hidden=768)
# caption_proj.0.weight: (caption_dim, bert_hidden=768)
#
ckpt_fused_dim  = sd["classifier.0.weight"].shape[1]   # e.g. 384 or 400
ckpt_hidden_dim = sd["itm_proj.0.weight"].shape[0]     # e.g. 256
ckpt_caption_dim = sd["caption_proj.0.weight"].shape[0] # e.g. 112 or 128
ckpt_fairface_out = ckpt_fused_dim - ckpt_hidden_dim - ckpt_caption_dim  # always 16

print(f"\nDims inferred from checkpoint weights:")
print(f"  classifier input (fused_dim) : {ckpt_fused_dim}")
print(f"  itm_proj output  (hidden_dim): {ckpt_hidden_dim}")
print(f"  caption_proj out (caption_dim): {ckpt_caption_dim}")
print(f"  fairface out     (fixed)     : {ckpt_fairface_out}")

# Warn if the notebook config differs from what was actually trained
if ckpt_hidden_dim != HIDDEN_DIM:
    print(f"  ⚠ HIDDEN_DIM mismatch: config={HIDDEN_DIM}, checkpoint={ckpt_hidden_dim}. Using checkpoint value.")
if ckpt_caption_dim != CAPTION_DIM:
    print(f"  ⚠ CAPTION_DIM mismatch: config={CAPTION_DIM}, checkpoint={ckpt_caption_dim}. Using checkpoint value.")

# ── Visual encoder (ResNet-50, frozen during training, not in checkpoint) ─────
visual_encoder = ResNetRegionExtractor(
    num_regions=MAX_VISUAL_LEN,
    local_weights_path=RESNET_WEIGHTS,
).to(DEVICE).eval()

# ── Rebuild classifier with exact dims from the checkpoint ───────────────────
model = UNITERITMClassifier(
    pretrained_name=PRETRAINED_NAME,
    visual_feat_dim=2048,
    fairface_dim=8,
    hidden_dim=ckpt_hidden_dim,    # ← from checkpoint, not config
    # caption_dim=ckpt_caption_dim,  # ← from checkpoint, not config
    dropout=DROPOUT,
    freeze_layers=FREEZE_LAYERS,
    max_visual_len=MAX_VISUAL_LEN,
).to(DEVICE)

model.load_state_dict(sd)
model.eval()
print("\nModel weights loaded successfully.")

# Print training args stored in checkpoint if available
if "args" in ckpt:
    print("\nTraining args stored in checkpoint:")
    for k, v in ckpt["args"].items():
        print(f"  {k}: {v}")

C:\Users\rkb1u25\AppData\Local\Temp\ipykernel_31600\713191456.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Checkpoint from epoch 4 | val_f1=0.6519


KeyError: 'itm_proj.0.weight'

## Cell 4 — Build Test DataLoader

In [ ]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
test_captions_df = pd.read_csv(TEST_CAPTIONS_CSV)   # must have: filename, text, label
objects_df       = pd.read_csv(TEST_OBJECTS_CSV)
faces_df         = pd.read_csv(TEST_FACES_CSV)
desc_df          = pd.read_csv(TEST_DESC_CSV)        # must have: filename, desc

print(f"Test samples : {len(test_captions_df)}")
print(f"Label distribution:\n{test_captions_df['label'].value_counts()}")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_NAME)

# ── Image transform (same val transform used during training) ─────────────────
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Dataset & DataLoader ──────────────────────────────────────────────────────
test_dataset = UNITERBinaryDataset(
    captions_df    = test_captions_df,
    desc_df        = desc_df,
    objects_df     = objects_df,
    faces_df       = faces_df,
    image_dir      = IMAGE_DIR,
    tokenizer      = tokenizer,
    image_transform= val_transform,
    max_text_len   = MAX_TEXT_LEN,
    max_desc_len   = MAX_DESC_LEN,
)

test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = (DEVICE.type == "cuda"),
)

print(f"Batches: {len(test_loader)}")

## Cell 5 — Run Inference

In [ ]:
all_labels    = []
all_probs     = []   # sigmoid probabilities for class 1 (good)
all_preds     = []   # hard predictions at THRESHOLD
all_filenames = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        pixel_values        = batch["pixel_values"].to(DEVICE)
        desc_input_ids      = batch["desc_input_ids"].to(DEVICE)
        desc_attention_mask = batch["desc_attention_mask"].to(DEVICE)
        cap_input_ids       = batch["cap_input_ids"].to(DEVICE)
        cap_attention_mask  = batch["cap_attention_mask"].to(DEVICE)
        fairface_vec        = batch["fairface_vec"].to(DEVICE)
        labels              = batch["label"]

        # Extract region features (ResNet frozen, no grad needed)
        visual_embeds, visual_mask = visual_encoder(pixel_values)

        # Forward pass
        logits = model(
            desc_input_ids        = desc_input_ids,
            desc_attention_mask   = desc_attention_mask,
            visual_embeds         = visual_embeds,
            visual_attention_mask = visual_mask,
            cap_input_ids         = cap_input_ids,
            cap_attention_mask    = cap_attention_mask,
            fairface_vec          = fairface_vec,
        )  # (B,) raw logits

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= THRESHOLD).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_filenames.extend(batch["filename"])

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print(f"Inference complete — {len(all_labels)} samples")

## Cell 6 — Metrics

In [ ]:
acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall    = recall_score(all_labels, all_preds, zero_division=0)
f1        = f1_score(all_labels, all_preds, zero_division=0)
auc       = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")
cm        = confusion_matrix(all_labels, all_preds)

print("=" * 45)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1        : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print("=" * 45)

print("\nFull classification report:")
print(classification_report(all_labels, all_preds, target_names=["bad (0)", "good (1)"]))

## Cell 7 — Confusion Matrix Plot

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Pred: bad", "Pred: good"],
    yticklabels=["True: bad", "True: good"],
    ax=ax,
)
ax.set_title("Confusion Matrix — UNITER/VILLA-ITM v2")
plt.tight_layout()
plt.show()

## Cell 8 — Probability Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label_val, label_name, color in [(0, "bad (0)", "tomato"), (1, "good (1)", "steelblue")]:
    mask = all_labels == label_val
    ax.hist(all_probs[mask], bins=30, alpha=0.6, label=label_name, color=color)
ax.axvline(THRESHOLD, color="black", linestyle="--", label=f"threshold={THRESHOLD}")
ax.set_xlabel("P(good)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class")
ax.legend()
plt.tight_layout()
plt.show()

## Cell 9 — Per-sample Results DataFrame

Useful for inspecting errors or exporting predictions.

In [ ]:
results_df = pd.DataFrame({
    "filename"   : all_filenames,
    "true_label" : all_labels,
    "pred_label" : all_preds,
    "prob_good"  : all_probs,
    "correct"    : (all_labels == all_preds),
})

print(f"Errors: {(~results_df['correct']).sum()} / {len(results_df)}")
results_df.head(10)

In [ ]:
# ── Optional: save predictions to CSV ─────────────────────────────────────────
OUTPUT_CSV = os.path.expanduser("~/img_logs/uniter_v2_test_predictions.csv")
results_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to {OUTPUT_CSV}")